# Thực nghiệm Tuần 8: Kiểm chứng tái lập Benchmark (#19) & Sửa lỗi Mask/Pose/Occlusion + Độ trễ CatVTON (#20)

Mục tiêu notebook:
1. Kiểm chứng tính tái lập khi khóa `pair_seed`, `cudnn.deterministic=True`, `cudnn.benchmark=False` và manifest có SHA-256 (`reproducibility_check.csv`).
2. Đối sánh trực tiếp trên 20 cặp cố định (`standard`, `mask_boundary`, `occlusion`, `pose_overall`) giữa:
   - `C0_baseline_s20_g45`: `AutoMasker` gốc + `blur=9`, 20 bước, `guidance=4.5`, `768x1024`.
   - `C1_refined_s20_g45`: `refined_cloth_agnostic_mask` + `preprocess_cloth_for_tryon`, 20 bước, `guidance=4.5`, `768x1024`.
   - `C2_refined_s12_g25`: `refined` mask, 12 bước, `guidance=2.5`, `768x1024`.
3. Khảo sát giới hạn tốc độ (`C3_refined_s15_g10` tắt CFG và `C4_refined_512_s20` độ phân giải `512x768`) trên tập phân tầng 8 cặp.
4. Tính toán thống nhất `LPIPS_out`, `SSIM_out`, `PSNR_out`, `CLIP_gain`, `DINO_gain`, và chỉ số bảo toàn vùng che khuất (`saved_region_ssim`), xuất bảng và ảnh đối sánh.


In [ ]:
# Cell 1: Cai dat thu vien, build detectron2, tai CatVTON (HF Space) va luu env_manifest.json
import os, sys, glob, json, shutil, subprocess, platform
os.chdir("/kaggle/working")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub>=0.34.0,<2.0", "diffusers>=0.30.0,<0.33.0",
    "transformers>=4.44.0,<4.50.0", "accelerate>=0.33.0,<1.2.0"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "fvcore", "iopath", "yacs", "omegaconf", "cloudpickle", "pycocotools", "av",
    "lpips", "torch-fidelity", "ninja", "wheel", "matplotlib", "tabulate",
    "termcolor", "scipy", "scikit-image"])

D2_GIT = "git+https://github.com/facebookresearch/detectron2.git"
WHEEL_DIR = "/kaggle/working/wheels"
os.makedirs(WHEEL_DIR, exist_ok=True)

def sh(cmd, env=None):
    p = subprocess.run(cmd, env=env, capture_output=True, text=True)
    return p.returncode == 0, (p.stdout + p.stderr)[-2500:]

def d2_ok():
    return sh([sys.executable, "-c", "from detectron2.config import get_cfg; from detectron2.engine.defaults import DefaultPredictor"])

def d2_install_wheel(path):
    ok, _ = sh([sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", path])
    return ok and d2_ok()[0]

def ensure_detectron2():
    if d2_ok()[0]:
        print("[detectron2] co san")
        return
    wheels = sorted(glob.glob("/kaggle/input/**/detectron2*.whl", recursive=True) + glob.glob(f"{WHEEL_DIR}/detectron2*.whl"))
    for w in wheels:
        if d2_install_wheel(w):
            print("[detectron2] cai tu wheel:", w)
            return
    env = {**os.environ, "MAX_JOBS": "4", "GIT_TERMINAL_PROMPT": "0"}
    env.pop("FORCE_CUDA", None)
    for name, e in [("auto", env), ("CPU-only", {**env, "CUDA_VISIBLE_DEVICES": ""})]:
        print(f"[detectron2] build ({name})...")
        ok, log = sh([sys.executable, "-m", "pip", "wheel", "--no-build-isolation", "--no-deps", "-w", WHEEL_DIR, D2_GIT], e)
        if not ok:
            print(log[-1000:])
            continue
        new = max(glob.glob(f"{WHEEL_DIR}/detectron2*.whl"), key=os.path.getmtime)
        if d2_install_wheel(new):
            print("[detectron2] build OK:", new)
            return
    raise RuntimeError("Khong cai duoc detectron2")

ensure_detectron2()

from huggingface_hub import snapshot_download
if not os.path.isdir("/kaggle/working/CatVTON"):
    snapshot_download(repo_type="space", repo_id="zhengchong/CatVTON", local_dir="/kaggle/working/CatVTON", ignore_patterns=["resource/*", "*.pyc"])

os.chdir("/kaggle/working/CatVTON")
for f in ["utils.py", "model/pipeline.py", "model/cloth_masker.py"]:
    assert os.path.isfile(f), f"Thieu {f}"

import torch, diffusers, transformers
OUT_DIR = "/kaggle/working/w08_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
env_info = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "diffusers": diffusers.__version__,
    "transformers": transformers.__version__,
    "gpu_count": torch.cuda.device_count(),
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}
with open(os.path.join(OUT_DIR, "env_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(env_info, f, indent=2)
print("Env:", env_info)


In [ ]:
# Cell 2: Doc frozen manifest, kiem tra SHA-256, nap CatVTON + AutoMasker + Refiner
import os, glob, json, hashlib, random, time, gc, math, io, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from diffusers import DPMSolverMultistepScheduler
from diffusers.image_processor import VaeImageProcessor
from huggingface_hub import snapshot_download
from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline
from utils import init_weight_dtype, resize_and_crop, resize_and_padding

def set_deterministic(seed: int, strict: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = strict
    torch.backends.cudnn.benchmark = not strict

set_deterministic(42, strict=True)

m_files = glob.glob("/kaggle/input/**/frozen_pairs_w08.json", recursive=True)
assert m_files, f"Khong thay frozen_pairs_w08.json trong /kaggle/input: {os.listdir('/kaggle/input')}"
MANIFEST_PATH = m_files[0]
DS_DIR = os.path.dirname(MANIFEST_PATH)
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    MANIFEST = json.load(f)

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for ch in iter(lambda: f.read(65536), b""):
            h.update(ch)
    return h.hexdigest()

for rec in MANIFEST["pairs"]:
    pf = os.path.join(DS_DIR, rec["person_file"])
    cf = os.path.join(DS_DIR, rec["cloth_file"])
    assert file_sha256(pf) == rec["person_sha256"], f"Sai SHA256 person: {rec['pair_id']}"
    assert file_sha256(cf) == rec["cloth_sha256"], f"Sai SHA256 cloth: {rec['pair_id']}"
print(f"[SHA-256 OK] Da xac minh {len(MANIFEST['pairs'])} cap trong {DS_DIR}")

"""
Bo tien xu ly va hieu chinh mat na CatVTON cho các ca pose, mask va occlusion (Issue #20).
Giu duoc ca luong fallback theo PIL/scipy khi chay kiem thu cuc bo khong co opencv.
"""

from typing import Union
import numpy as np
from PIL import Image, ImageFilter

try:
    import cv2  # type: ignore
except ImportError:
    cv2 = None

from scipy import ndimage  # type: ignore

DENSE_INDEX_MAP = {
    "background": [0],
    "torso": [1, 2],
    "right hand": [3],
    "left hand": [4],
    "right foot": [5],
    "left foot": [6],
    "right thigh": [7, 9],
    "left thigh": [8, 10],
    "right leg": [11, 13],
    "left leg": [12, 14],
    "left big arm": [15, 17],
    "right big arm": [16, 18],
    "left forearm": [19, 21],
    "right forearm": [20, 22],
    "face": [23, 24],
    "thighs": [7, 8, 9, 10],
    "legs": [11, 12, 13, 14],
    "hands": [3, 4],
    "feet": [5, 6],
    "big arms": [15, 16, 17, 18],
    "forearms": [19, 20, 21, 22],
}

ATR_MAPPING = {
    "Background": 0, "Hat": 1, "Hair": 2, "Sunglasses": 3,
    "Upper-clothes": 4, "Skirt": 5, "Pants": 6, "Dress": 7,
    "Belt": 8, "Left-shoe": 9, "Right-shoe": 10, "Face": 11,
    "Left-leg": 12, "Right-leg": 13, "Left-arm": 14, "Right-arm": 15,
    "Bag": 16, "Scarf": 17,
}

LIP_MAPPING = {
    "Background": 0, "Hat": 1, "Hair": 2, "Glove": 3,
    "Sunglasses": 4, "Upper-clothes": 5, "Dress": 6, "Coat": 7,
    "Socks": 8, "Pants": 9, "Jumpsuits": 10, "Scarf": 11,
    "Skirt": 12, "Face": 13, "Left-arm": 14, "Right-arm": 15,
    "Left-leg": 16, "Right-leg": 17, "Left-shoe": 18, "Right-shoe": 19,
}

PROTECT_BODY_PARTS = {
    "upper": ["Left-leg", "Right-leg"],
    "lower": ["Right-arm", "Left-arm", "Face"],
    "overall": [],
    "inner": ["Left-leg", "Right-leg"],
    "outer": ["Left-leg", "Right-leg"],
}

PROTECT_CLOTH_PARTS = {
    "upper": {"ATR": ["Skirt", "Pants"], "LIP": ["Skirt", "Pants"]},
    "lower": {"ATR": ["Upper-clothes"], "LIP": ["Upper-clothes", "Coat"]},
    "overall": {"ATR": [], "LIP": []},
    "inner": {"ATR": ["Dress", "Coat", "Skirt", "Pants"], "LIP": ["Dress", "Coat", "Skirt", "Pants", "Jumpsuits"]},
    "outer": {"ATR": ["Dress", "Pants", "Skirt"], "LIP": ["Upper-clothes", "Dress", "Pants", "Skirt", "Jumpsuits"]},
}

# Goc trong CatVTON de upper chua ca Dress/Jumpsuits gay tran mask quan toi mau;
# ban refined chi lay Upper-clothes va Coat cho upper.
REFINED_MASK_CLOTH_PARTS = {
    "upper": ["Upper-clothes", "Coat"],
    "lower": ["Pants", "Skirt"],
    "overall": ["Upper-clothes", "Dress", "Pants", "Skirt", "Coat", "Jumpsuits"],
    "inner": ["Upper-clothes"],
    "outer": ["Coat"],
}

MASK_DENSE_PARTS = {
    "upper": ["torso", "big arms", "forearms"],
    "lower": ["thighs", "legs"],
    "overall": ["torso", "thighs", "legs", "big arms", "forearms"],
    "inner": ["torso"],
    "outer": ["torso", "big arms", "forearms"],
}


def part_mask_of(part: Union[str, list], parse: np.ndarray, mapping: dict) -> np.ndarray:
    if isinstance(part, str):
        part = [part]
    mask = np.zeros_like(parse, dtype=bool)
    for name in part:
        if name not in mapping:
            continue
        idx = mapping[name]
        if isinstance(idx, list):
            for i in idx:
                mask |= (parse == i)
        else:
            mask |= (parse == idx)
    return mask


def _dilate(mask_bool: np.ndarray, ksize: int, iterations: int = 1) -> np.ndarray:
    if ksize <= 1 or iterations <= 0:
        return mask_bool
    if cv2 is not None:
        kernel = np.ones((ksize, ksize), np.uint8)
        out = cv2.dilate(mask_bool.astype(np.uint8), kernel, iterations=iterations)
        return out > 0
    struct = np.ones((ksize, ksize), dtype=bool)
    return ndimage.binary_dilation(mask_bool, structure=struct, iterations=iterations)


def _close_holes(mask_bool: np.ndarray, ksize: int) -> np.ndarray:
    """Dong cac lo nho ben trong than ao ma khong bac cau qua khoang trong khuyu tay."""
    if ksize <= 1:
        return mask_bool
    if cv2 is not None:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
        closed = cv2.morphologyEx(mask_bool.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
        return closed > 0
    yy, xx = np.ogrid[-ksize // 2 : ksize // 2 + 1, -ksize // 2 : ksize // 2 + 1]
    struct = (xx * xx + yy * yy) <= (ksize / 2.0) ** 2
    return ndimage.binary_closing(mask_bool, structure=struct)


def _feather_and_threshold(mask_bool: np.ndarray, ksize: int, threshold: int = 100) -> np.ndarray:
    if ksize <= 1:
        return mask_bool
    if cv2 is not None:
        blurred = cv2.GaussianBlur(mask_bool.astype(np.uint8) * 255, (ksize, ksize), 0)
        return blurred >= threshold
    sigma = max(1.0, ksize / 4.0)
    blurred = ndimage.gaussian_filter(mask_bool.astype(np.float32) * 255.0, sigma=sigma)
    return blurred >= threshold


def refined_cloth_agnostic_mask(
    densepose_mask: Image.Image,
    schp_lip_mask: Image.Image,
    schp_atr_mask: Image.Image,
    part: str = "upper",
) -> Image.Image:
    """
    Tao mat na da hieu chinh cho CatVTON:
    1. Bo convex hull (hull_mask) khi part != 'overall' de tranh lap kin lo chong tay/tui xach.
    2. Giam nhan lam mo vien va tru lai toan bo vung bao ve (toc, tui, ban tay, quan) sau cung.
    3. Tach Dress/Jumpsuits khoi strong_mask cua upper va chan bien duoi theo khop dui DensePose.
    """
    assert part in ["upper", "lower", "overall", "inner", "outer"], f"Invalid part: {part}"
    w, h = densepose_mask.size

    dp = np.array(densepose_mask)
    lip = np.array(schp_lip_mask)
    atr = np.array(schp_atr_mask)

    dilate_k = max(3, max(w, h) // 250)
    if dilate_k % 2 == 0:
        dilate_k += 1

    # Nhan dong nho (~1.5% chieu cao) chi han khe ho vai, khong bac cau qua khuyu tay (~10-20% anh)
    close_k = max(7, min(19, max(w, h) // 64))
    if close_k % 2 == 0:
        close_k += 1

    # Lam min mep cuc bo thay vi kernel 41x41 cua ban goc
    blur_k = max(5, min(13, max(w, h) // 90))
    if blur_k % 2 == 0:
        blur_k += 1

    # 1. Vung bao ve manh: mat + ban tay (DensePose hands khong bat buoc giao rong voi SCHP) + tui/phu kien
    dp_hands = part_mask_of(["hands", "feet"], dp, DENSE_INDEX_MAP)
    dp_hands = _dilate(dp_hands, dilate_k, iterations=1)
    schp_arms = part_mask_of(["Left-arm", "Right-arm"], atr, ATR_MAPPING) | part_mask_of(["Left-arm", "Right-arm"], lip, LIP_MAPPING)

    # Neu tay cam vat the (tui/coc), DensePose van giu duoc vung ban tay ngay ca khi SCHP bi nham
    hands_protect = dp_hands | (dp_hands & schp_arms)
    face_protect = part_mask_of("Face", lip, LIP_MAPPING) | part_mask_of("Face", atr, ATR_MAPPING)
    bag_protect = part_mask_of(["Bag", "Glove"], atr, ATR_MAPPING) | part_mask_of(["Glove"], lip, LIP_MAPPING)
    bag_protect = _dilate(bag_protect, dilate_k, iterations=1)

    strong_protect = hands_protect | face_protect | bag_protect

    # 2. Vung bao ve toc, phu kien va trang phuc khong thuoc vung thu
    body_protect = part_mask_of(PROTECT_BODY_PARTS[part], lip, LIP_MAPPING) | part_mask_of(PROTECT_BODY_PARTS[part], atr, ATR_MAPPING)
    hair_protect = part_mask_of(["Hair"], lip, LIP_MAPPING) | part_mask_of(["Hair"], atr, ATR_MAPPING)
    cloth_protect = (
        part_mask_of(PROTECT_CLOTH_PARTS[part]["LIP"], lip, LIP_MAPPING)
        | part_mask_of(PROTECT_CLOTH_PARTS[part]["ATR"], atr, ATR_MAPPING)
    )
    acc_parts = ["Hat", "Sunglasses", "Left-shoe", "Right-shoe", "Scarf", "Socks"]
    acc_protect = part_mask_of(acc_parts, lip, LIP_MAPPING) | part_mask_of(acc_parts, atr, ATR_MAPPING)

    all_protect = strong_protect | body_protect | hair_protect | cloth_protect | acc_protect

    # 3. Vung mat na muc tieu
    strong_mask = (
        part_mask_of(REFINED_MASK_CLOTH_PARTS[part], lip, LIP_MAPPING)
        | part_mask_of(REFINED_MASK_CLOTH_PARTS[part], atr, ATR_MAPPING)
    )
    background = part_mask_of(["Background"], lip, LIP_MAPPING) & part_mask_of(["Background"], atr, ATR_MAPPING)
    mask_dense = part_mask_of(MASK_DENSE_PARTS[part], dp, DENSE_INDEX_MAP)
    mask_dense = _dilate(mask_dense, dilate_k, iterations=2)

    # Bo sung vung co the tren ao + DensePose (nhung giu nguyen vung nen trong giua tay chong hong)
    mask_area = ((~all_protect) & (~background) & (strong_mask | mask_dense)) | (mask_dense & (~all_protect))

    # Chi dung morphological closing nho, khong dung convex hull cho upper
    mask_area = _close_holes(mask_area, close_k)
    mask_area = _feather_and_threshold(mask_area, blur_k, threshold=110)

    # Hop voi strong_mask (ao dai tay co tay hep de bi DensePose bo sot o co tay -> can strong_mask)
    mask_area = mask_area | strong_mask
    mask_area = _dilate(mask_area, dilate_k, iterations=1)

    # 4. Chan bien duoi khi part == 'upper' dua vao vung dui DensePose de tranh tran mask khi ao/quan den tiep mau
    if part == "upper":
        thighs = part_mask_of(["thighs", "legs"], dp, DENSE_INDEX_MAP)
        thigh_ys = np.where(thighs)[0]
        if len(thigh_ys) >= max(50, int(0.001 * w * h)):
            y_hip_cut = int(np.percentile(thigh_ys, 5)) + int(0.025 * h)
            # Chi cho phep giu phan duoi y_hip_cut neu ca LIP va ATR deu xac nhan la Upper-clothes
            strict_upper = part_mask_of(["Upper-clothes"], lip, LIP_MAPPING) & part_mask_of(["Upper-clothes"], atr, ATR_MAPPING)
            below_cut = np.zeros_like(mask_area, dtype=bool)
            below_cut[min(y_hip_cut, h) :, :] = True
            mask_area = mask_area & (~(below_cut & (~strict_upper)))

    # 5. Tru lai toan bo vung bao ve o buoc CUOI CUNG sau khi blur va dilate
    final_mask = mask_area & (~all_protect)

    return Image.fromarray((final_mask.astype(np.uint8)) * 255)


def preprocess_cloth_for_tryon(
    cloth_img: Image.Image,
    target_size: tuple[int, int] = (768, 1024),
    neutral_bg: tuple[int, int, int] = (232, 230, 224),
) -> tuple[Image.Image, bool]:
    """
    Chuan hoa kich thuoc va kiem tra nguy co ao trang tren nen trang.
    Neu goc anh la nen trang (>240) va phan trang phuc cung rat sang (>232),
    dem nen xam trung tinh nhat de Reference UNet bat duoc duong vien ao.
    """
    rgb = cloth_img.convert("RGB")
    arr = np.asarray(rgb, dtype=np.float32)
    h, w, _ = arr.shape
    patch = max(4, min(h, w) // 20)
    corners = np.concatenate([
        arr[:patch, :patch].reshape(-1, 3),
        arr[:patch, -patch:].reshape(-1, 3),
        arr[-patch:, :patch].reshape(-1, 3),
        arr[-patch:, -patch:].reshape(-1, 3),
    ], axis=0)
    corner_mean = float(corners.mean())

    # Uoc luong do sang vung giua (trang phuc)
    center = arr[h // 4 : (3 * h) // 4, w // 4 : (3 * w) // 4]
    center_bright_ratio = float((center.mean(axis=2) > 228.0).mean())

    white_on_white = (corner_mean > 240.0) and (center_bright_ratio > 0.60)
    bg_color = neutral_bg if white_on_white else (255, 255, 255)

    tw, th = target_size
    scale = min(tw / w, th / h)
    nw, nh = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    resized = rgb.resize((nw, nh), Image.LANCZOS)

    canvas = Image.new("RGB", (tw, th), bg_color)
    canvas.paste(resized, ((tw - nw) // 2, (th - nh) // 2))

    # Neu ao trang nam tren nen trang san ben trong crop -> thay nen ngoai vien ao bang neutral_bg
    if white_on_white:
        c_arr = np.asarray(canvas, dtype=np.uint8).copy()
        is_pure_white = (c_arr[:, :, 0] > 246) & (c_arr[:, :, 1] > 246) & (c_arr[:, :, 2] > 246)
        # Chi floodfill tu 4 mep ngoai de khong doi mau ben trong mat ao
        seed = np.zeros((th, tw), dtype=bool)
        seed[0, :] = is_pure_white[0, :]
        seed[-1, :] = is_pure_white[-1, :]
        seed[:, 0] = is_pure_white[:, 0]
        seed[:, -1] = is_pure_white[:, -1]
        bg_mask = ndimage.binary_propagation(seed, mask=is_pure_white)
        c_arr[bg_mask] = neutral_bg
        canvas = Image.fromarray(c_arr)

    return canvas, white_on_white


def find_local_dir(root, folder_name):
    if not os.path.isdir(root):
        return None
    for m in glob.glob(os.path.join(root, "**", folder_name), recursive=True):
        if os.path.isdir(m) and os.listdir(m):
            return m
    return None

DATASET_ROOT = "/kaggle/input/catvton-weights"
WORKING_ROOT = "/kaggle/working/catvton_weights"
catvton_local = find_local_dir(DATASET_ROOT, "CatVTON") or os.path.join(WORKING_ROOT, "CatVTON")
repo_path = catvton_local if (os.path.isdir(catvton_local) and os.listdir(catvton_local)) else snapshot_download(repo_id="zhengchong/CatVTON", local_dir=catvton_local)
sd_local = find_local_dir(DATASET_ROOT, "sd-inpainting") or os.path.join(WORKING_ROOT, "sd-inpainting")
base_ckpt_path = sd_local if (os.path.isdir(sd_local) and os.listdir(sd_local)) else snapshot_download(repo_id="booksforcharlie/stable-diffusion-inpainting", local_dir=sd_local)

t_cold0 = time.perf_counter()
WORKERS = []
n_gpus = min(2, torch.cuda.device_count()) if torch.cuda.is_available() else 0
assert n_gpus >= 1, "Can it nhat 1 GPU"
for g in range(n_gpus):
    dev_str = f"cuda:{g}"
    pl = CatVTONPipeline(
        base_ckpt=base_ckpt_path,
        attn_ckpt=repo_path,
        attn_ckpt_version="mix",
        weight_dtype=init_weight_dtype("fp16"),
        use_tf32=False,
        device=dev_str,
        skip_safety_check=True,
    )
    pl.noise_scheduler = DPMSolverMultistepScheduler.from_config(
        pl.noise_scheduler.config, use_karras_sigmas=True, algorithm_type="dpmsolver++"
    )
    am = AutoMasker(
        densepose_ckpt=os.path.join(repo_path, "DensePose"),
        schp_ckpt=os.path.join(repo_path, "SCHP"),
        device=dev_str,
    )
    WORKERS.append((pl, am, g))
mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)
torch.cuda.synchronize()
COLD_START_S = round(time.perf_counter() - t_cold0, 3)
print(f"[OK] Nap {len(WORKERS)} worker GPU trong {COLD_START_S}s")


In [ ]:
# Cell 3: Thuc nghiem 1 (Issue #19) - Kiem tra tinh tai lap (Deterministic vs Non-deterministic)
from skimage.metrics import structural_similarity as ssim_fn

TMP_ROOT = "/tmp/w08_eval"
shutil.rmtree(TMP_ROOT, ignore_errors=True)
os.makedirs(TMP_ROOT, exist_ok=True)
OUT_DIR = "/kaggle/working/w08_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

PRE_CACHE = {}

def run_single_pair(rec, worker, mask_mode="refined", steps=20, guidance=4.5, width=768, height=1024, deterministic=True, use_pre_cache=True):
    pl, am, gpu = worker
    pid = rec["pair_id"]
    p_path = os.path.join(DS_DIR, rec["person_file"])
    c_path = os.path.join(DS_DIR, rec["cloth_file"])
    person_raw = Image.open(p_path).convert("RGB")
    cloth_raw = Image.open(c_path).convert("RGB")

    with torch.cuda.device(gpu):
        if deterministic:
            set_deterministic(rec["pair_seed"], strict=True)
            gen = torch.Generator(device=f"cuda:{gpu}").manual_seed(rec["pair_seed"])
        else:
            torch.backends.cudnn.deterministic = False
            torch.backends.cudnn.benchmark = True
            gen = None

        torch.cuda.reset_peak_memory_stats(gpu)
        torch.cuda.synchronize(gpu)
        t0 = time.perf_counter()

        person_img = resize_and_crop(person_raw, (width, height))
        white_trig = False
        if mask_mode == "refined":
            cloth_img, white_trig = preprocess_cloth_for_tryon(cloth_raw, target_size=(width, height))
        else:
            cloth_img = resize_and_padding(cloth_raw, (width, height))

        cache_key = (pid, width, height, gpu)
        cache_hit = False
        t_am0 = time.perf_counter()
        if use_pre_cache and cache_key in PRE_CACHE:
            pre_res = PRE_CACHE[cache_key]
            cache_hit = True
        else:
            pre_res = am.preprocess_image(person_img)
            PRE_CACHE[cache_key] = pre_res
        torch.cuda.synchronize(gpu)
        t_am_s = round(time.perf_counter() - t_am0, 4)

        if mask_mode == "refined":
            raw_mask = refined_cloth_agnostic_mask(pre_res["densepose"], pre_res["schp_lip"], pre_res["schp_atr"], part=rec["category"])
            blur_mask = mask_processor.blur(raw_mask, blur_factor=5)
        else:
            raw_mask = am.cloth_agnostic_mask(pre_res["densepose"], pre_res["schp_lip"], pre_res["schp_atr"], part=rec["category"])
            blur_mask = mask_processor.blur(raw_mask, blur_factor=9)
        torch.cuda.synchronize(gpu)
        t_pre_total = round(time.perf_counter() - t0, 4)

        t_inf0 = time.perf_counter()
        with torch.inference_mode():
            out_img = pl(
                image=person_img,
                condition_image=cloth_img,
                mask=blur_mask,
                num_inference_steps=steps,
                guidance_scale=guidance,
                width=width,
                height=height,
                generator=gen,
            )[0]
        torch.cuda.synchronize(gpu)
        t_inf = round(time.perf_counter() - t_inf0, 4)
        if out_img.size != (width, height):
            out_img = out_img.resize((width, height), Image.LANCZOS)

        t_total = round(time.perf_counter() - t0, 4)
        peak_vram = round(torch.cuda.max_memory_allocated(gpu) / (1024 ** 3), 3)
        mask_pct = round(float((np.asarray(raw_mask) > 127).mean() * 100.0), 2)

    return {
        "person_img": person_img,
        "cloth_img": cloth_img,
        "raw_mask": raw_mask,
        "out_img": out_img,
        "t_automask_s": t_am_s,
        "cache_hit": cache_hit,
        "t_preprocess_s": t_pre_total,
        "t_infer_s": t_inf,
        "t_total_s": t_total,
        "peak_vram_gb": peak_vram,
        "mask_area_pct": mask_pct,
        "white_cloth_trig": white_trig,
    }

def img_diff_metrics(im1, im2):
    a = np.asarray(im1, dtype=np.float32)
    b = np.asarray(im2, dtype=np.float32)
    diff = np.abs(a - b)
    mae = float(diff.mean())
    max_abs = float(diff.max())
    exact_pct = float((diff.max(axis=2) == 0).mean() * 100.0)
    sv = float(ssim_fn(a.astype(np.uint8), b.astype(np.uint8), channel_axis=2, data_range=255))
    return round(mae, 5), round(max_abs, 2), round(exact_pct, 2), round(sv, 6)

REPRO_IDS = ["S01", "M01", "O01"]
repro_pairs = [r for r in MANIFEST["pairs"] if r["pair_id"] in REPRO_IDS]
repro_rows = []
repro_vis = []

for rec in repro_pairs:
    pid = rec["pair_id"]
    print(f"[REPRO] Testing {pid} (seed={rec['pair_seed']})...")
    resA = run_single_pair(rec, WORKERS[0], mask_mode="refined", deterministic=True, use_pre_cache=False)
    torch.cuda.empty_cache()
    resB = run_single_pair(rec, WORKERS[0], mask_mode="refined", deterministic=True, use_pre_cache=False)
    torch.cuda.empty_cache()
    resD = run_single_pair(rec, WORKERS[0], mask_mode="refined", deterministic=False, use_pre_cache=False)

    mae_ab, max_ab, ex_ab, ssim_ab = img_diff_metrics(resA["out_img"], resB["out_img"])
    mae_ad, max_ad, ex_ad, ssim_ad = img_diff_metrics(resA["out_img"], resD["out_img"])
    repro_rows.append({
        "pair_id": pid, "comparison": "Det_RunA_vs_Det_RunB_GPU0",
        "mae_px": mae_ab, "max_abs_px": max_ab, "exact_pixel_pct": ex_ab, "ssim": ssim_ab,
    })
    repro_rows.append({
        "pair_id": pid, "comparison": "Det_RunA_vs_NoSeed_RunD_GPU0",
        "mae_px": mae_ad, "max_abs_px": max_ad, "exact_pixel_pct": ex_ad, "ssim": ssim_ad,
    })

    if len(WORKERS) > 1:
        resC = run_single_pair(rec, WORKERS[1], mask_mode="refined", deterministic=True, use_pre_cache=False)
        mae_ac, max_ac, ex_ac, ssim_ac = img_diff_metrics(resA["out_img"], resC["out_img"])
        repro_rows.append({
            "pair_id": pid, "comparison": "Det_RunA_GPU0_vs_Det_RunC_GPU1",
            "mae_px": mae_ac, "max_abs_px": max_ac, "exact_pixel_pct": ex_ac, "ssim": ssim_ac,
        })

    d_det = np.clip(np.abs(np.asarray(resA["out_img"], dtype=np.float32) - np.asarray(resB["out_img"], dtype=np.float32)) * 5.0, 0, 255).astype(np.uint8)
    d_nodet = np.clip(np.abs(np.asarray(resA["out_img"], dtype=np.float32) - np.asarray(resD["out_img"], dtype=np.float32)) * 5.0, 0, 255).astype(np.uint8)
    repro_vis.append((pid, resA["out_img"], resB["out_img"], resD["out_img"], Image.fromarray(d_det), Image.fromarray(d_nodet)))

df_repro = pd.DataFrame(repro_rows)
df_repro.to_csv(os.path.join(OUT_DIR, "reproducibility_check.csv"), index=False)
print(df_repro.to_string(index=False))

tw, th = 256, 341
canvas = Image.new("RGB", (tw * 5, th * len(repro_vis)), (255, 255, 255))
for ri, (pid, imA, imB, imD, d1, d2) in enumerate(repro_vis):
    for ci, im in enumerate([imA, imB, d1, imD, d2]):
        canvas.paste(im.resize((tw, th), Image.LANCZOS), (ci * tw, ri * th))
canvas.save(os.path.join(OUT_DIR, "fig_reproducibility_diff.jpg"), quality=90)


In [ ]:
# Cell 4: Thuc nghiem 2 (Issue #20) - Chay 5 cau hinh tren bo dong bang 20 cap (C0, C1, C2 full; C3, C4 phan tang)
from concurrent.futures import ThreadPoolExecutor

STRATIFIED_8 = {"S01", "S02", "M01", "M02", "O01", "O02", "P01", "P02"}

EXPERIMENTS = [
    {"name": "C0_baseline_s20_g45", "pairs": MANIFEST["pairs"], "mask_mode": "baseline", "steps": 20, "guidance": 4.5, "w": 768, "h": 1024, "use_cache": False},
    {"name": "C1_refined_s20_g45",  "pairs": MANIFEST["pairs"], "mask_mode": "refined",  "steps": 20, "guidance": 4.5, "w": 768, "h": 1024, "use_cache": True},
    {"name": "C2_refined_s12_g25",  "pairs": MANIFEST["pairs"], "mask_mode": "refined",  "steps": 12, "guidance": 2.5, "w": 768, "h": 1024, "use_cache": True},
    {"name": "C3_refined_s15_g10",  "pairs": [r for r in MANIFEST["pairs"] if r["pair_id"] in STRATIFIED_8], "mask_mode": "refined", "steps": 15, "guidance": 1.0, "w": 768, "h": 1024, "use_cache": True},
    {"name": "C4_refined_512_s20",  "pairs": [r for r in MANIFEST["pairs"] if r["pair_id"] in STRATIFIED_8], "mask_mode": "refined", "steps": 20, "guidance": 4.5, "w": 512, "h": 768,  "use_cache": True},
]

all_run_logs = []

for exp in EXPERIMENTS:
    ename = exp["name"]
    epairs = exp["pairs"]
    edir = os.path.join(TMP_ROOT, ename)
    for sub in ["person", "cloth", "mask", "gen"]:
        os.makedirs(os.path.join(edir, sub), exist_ok=True)
    print(f"=== START {ename} ({len(epairs)} pairs) ===")
    t_exp0 = time.perf_counter()

    def _run_shard(w_idx):
        worker = WORKERS[w_idx]
        shard = epairs[w_idx :: len(WORKERS)]
        rows = []
        for rec in shard:
            pid = rec["pair_id"]
            res = run_single_pair(
                rec, worker,
                mask_mode=exp["mask_mode"],
                steps=exp["steps"],
                guidance=exp["guidance"],
                width=exp["w"],
                height=exp["h"],
                deterministic=True,
                use_pre_cache=exp["use_cache"],
            )
            res["person_img"].save(os.path.join(edir, "person", f"{pid}.png"))
            res["cloth_img"].save(os.path.join(edir, "cloth", f"{pid}.png"))
            res["raw_mask"].convert("L").save(os.path.join(edir, "mask", f"{pid}.png"))
            res["out_img"].save(os.path.join(edir, "gen", f"{pid}.png"))
            rows.append({
                "exp_name": ename,
                "pair_id": pid,
                "group": rec["group"],
                "category": rec["category"],
                "mask_mode": exp["mask_mode"],
                "steps": exp["steps"],
                "guidance_scale": exp["guidance"],
                "resolution": f"{exp['w']}x{exp['h']}",
                "pair_seed": rec["pair_seed"],
                "t_automask_s": res["t_automask_s"],
                "mask_cache_hit": res["cache_hit"],
                "t_preprocess_s": res["t_preprocess_s"],
                "t_infer_s": res["t_infer_s"],
                "t_total_s": res["t_total_s"],
                "peak_vram_gb": res["peak_vram_gb"],
                "mask_area_pct": res["mask_area_pct"],
                "white_cloth_trig": res["white_cloth_trig"],
            })
        return rows

    with ThreadPoolExecutor(len(WORKERS)) as ex:
        shard_results = list(ex.map(_run_shard, range(len(WORKERS))))
    for sr in shard_results:
        all_run_logs.extend(sr)
    print(f"=== DONE {ename} in {time.perf_counter() - t_exp0:.1f}s ===")

df_runs = pd.DataFrame(all_run_logs)
df_runs.to_csv(os.path.join(OUT_DIR, "raw_timing_logs.csv"), index=False)
print(df_runs.groupby("exp_name")[["t_preprocess_s", "t_infer_s", "t_total_s", "peak_vram_gb", "mask_area_pct"]].mean().round(3))


In [ ]:
# Cell 5: Giai phong pipeline GPU -> Cham metric thong nhat + so sanh vung bao ve + tao anh ghep
del WORKERS
gc.collect()
torch.cuda.empty_cache()

import lpips
from skimage.metrics import structural_similarity, peak_signal_noise_ratio
from transformers import AutoModel, AutoImageProcessor, CLIPModel, CLIPImageProcessor

dev = "cuda:0"
lp_spatial = lpips.LPIPS(net="alex", spatial=True).to(dev).eval()
CLIP_ID = "openai/clip-vit-large-patch14"
DINO_ID = "facebook/dino-vits16"
clip_model = CLIPModel.from_pretrained(CLIP_ID).to(dev).eval()
clip_proc = CLIPImageProcessor.from_pretrained(CLIP_ID)
dino_model = AutoModel.from_pretrained(DINO_ID).to(dev).eval()
dino_proc = AutoImageProcessor.from_pretrained(DINO_ID)

def to_tensor(img):
    x = torch.from_numpy(np.asarray(img)).float().permute(2, 0, 1)[None] / 127.5 - 1.0
    return x.to(dev)

def dilate_torch(mask_bool, k=16):
    m = torch.from_numpy(mask_bool).float()[None, None].to(dev)
    if k > 0:
        m = F.max_pool2d(m, 2 * k + 1, stride=1, padding=k)
    return (m[0, 0] > 0.5).cpu().numpy()

def crop_box(mask_bool, pad=8):
    ys, xs = np.where(mask_bool)
    if len(ys) == 0:
        return None
    h, w = mask_bool.shape
    return (int(max(xs.min() - pad, 0)), int(max(ys.min() - pad, 0)), int(min(xs.max() + pad + 1, w)), int(min(ys.max() + pad + 1, h)))

def to_square(img, fill=255):
    s = max(img.size)
    c = Image.new("RGB", (s, s), (fill, fill, fill))
    c.paste(img, ((s - img.width) // 2, (s - img.height) // 2))
    return c

@torch.no_grad()
def embed_clip(imgs):
    px = clip_proc(images=[to_square(i) for i in imgs], return_tensors="pt")["pixel_values"].to(dev)
    v = clip_model.vision_model(pixel_values=px).pooler_output
    return F.normalize(clip_model.visual_projection(v).float(), dim=-1)

@torch.no_grad()
def embed_dino(imgs):
    px = dino_proc(images=[to_square(i) for i in imgs], return_tensors="pt")["pixel_values"].to(dev)
    f = dino_model(pixel_values=px).last_hidden_state[:, 0]
    return F.normalize(f.float(), dim=-1)

metric_rows = []
protected_rows = []

for row in all_run_logs:
    ename = row["exp_name"]
    pid = row["pair_id"]
    edir = os.path.join(TMP_ROOT, ename)
    c0_dir = os.path.join(TMP_ROOT, "C0_baseline_s20_g45")

    gen = Image.open(os.path.join(edir, "gen", f"{pid}.png")).convert("RGB")
    person = Image.open(os.path.join(edir, "person", f"{pid}.png")).convert("RGB")
    cloth = Image.open(os.path.join(edir, "cloth", f"{pid}.png")).convert("RGB")
    own_m = np.asarray(Image.open(os.path.join(edir, "mask", f"{pid}.png")).convert("L")) > 127

    base_m_pil = Image.open(os.path.join(c0_dir, "mask", f"{pid}.png")).convert("L").resize(gen.size, Image.NEAREST)
    base_m = np.asarray(base_m_pil) > 127

    keep_own = ~dilate_torch(own_m, 16)
    keep_base = ~dilate_torch(base_m, 16)

    with torch.no_grad():
        lmap = lp_spatial(to_tensor(gen), to_tensor(person))[0, 0].cpu().numpy()

    g_np, p_np = np.asarray(gen), np.asarray(person)
    _, smap = structural_similarity(g_np, p_np, channel_axis=2, data_range=255, full=True)
    sq_err = (g_np.astype(np.float32) - p_np.astype(np.float32)) ** 2

    def _masked_stats(keep):
        if keep.sum() == 0:
            return float("nan"), float("nan"), float("nan")
        lp = float(lmap[keep].mean())
        ss = float(smap[keep].mean())
        mse = float(sq_err[keep].mean())
        ps = 100.0 if mse == 0 else min(100.0, 10.0 * math.log10(255.0 ** 2 / mse))
        return round(lp, 4), round(ss, 4), round(ps, 2)

    lp_own, ss_own, ps_own = _masked_stats(keep_own)
    lp_base, ss_base, ps_base = _masked_stats(keep_base)

    box = crop_box(own_m)
    g_crop = gen.crop(box) if box else gen
    p_crop = person.crop(box) if box else person
    ec = embed_clip([g_crop, cloth, p_crop])
    ed = embed_dino([g_crop, cloth, p_crop])

    clip_gen = float((ec[0] * ec[1]).sum().item())
    clip_base = float((ec[2] * ec[1]).sum().item())
    dino_gen = float((ed[0] * ed[1]).sum().item())
    dino_base = float((ed[2] * ed[1]).sum().item())

    metric_rows.append({
        **row,
        "lpips_out_own": lp_own,
        "ssim_out_own": ss_own,
        "psnr_out_own": ps_own,
        "lpips_out_unified": lp_base,
        "ssim_out_unified": ss_base,
        "psnr_out_unified": ps_base,
        "clip_cloth": round(clip_gen, 4),
        "clip_base": round(clip_base, 4),
        "clip_gain": round(clip_gen - clip_base, 4),
        "dino_cloth": round(dino_gen, 4),
        "dino_base": round(dino_base, 4),
        "dino_gain": round(dino_gen - dino_base, 4),
    })

    if ename == "C1_refined_s20_g45":
        c0_gen = Image.open(os.path.join(c0_dir, "gen", f"{pid}.png")).convert("RGB")
        c0_np = np.asarray(c0_gen)
        saved_region = base_m & (~own_m)
        saved_pct = float(saved_region.mean() * 100.0)
        if saved_region.sum() >= 50:
            _, s_c0 = structural_similarity(c0_np, p_np, channel_axis=2, data_range=255, full=True)
            _, s_c1 = structural_similarity(g_np, p_np, channel_axis=2, data_range=255, full=True)
            with torch.no_grad():
                l_c0 = lp_spatial(to_tensor(c0_gen), to_tensor(person))[0, 0].cpu().numpy()
            ssim_saved_c0 = round(float(s_c0[saved_region].mean()), 4)
            ssim_saved_c1 = round(float(s_c1[saved_region].mean()), 4)
            lpips_saved_c0 = round(float(l_c0[saved_region].mean()), 4)
            lpips_saved_c1 = round(float(lmap[saved_region].mean()), 4)
        else:
            ssim_saved_c0 = ssim_saved_c1 = lpips_saved_c0 = lpips_saved_c1 = float("nan")
        protected_rows.append({
            "pair_id": pid,
            "group": row["group"],
            "saved_region_pct": round(saved_pct, 2),
            "c0_mask_pct": round(float(base_m.mean() * 100.0), 2),
            "c1_mask_pct": round(float(own_m.mean() * 100.0), 2),
            "ssim_saved_c0_baseline": ssim_saved_c0,
            "ssim_saved_c1_refined": ssim_saved_c1,
            "lpips_saved_c0_baseline": lpips_saved_c0,
            "lpips_saved_c1_refined": lpips_saved_c1,
        })

df_metrics = pd.DataFrame(metric_rows)
df_metrics.to_csv(os.path.join(OUT_DIR, "per_pair_metrics.csv"), index=False)

df_prot = pd.DataFrame(protected_rows)
df_prot.to_csv(os.path.join(OUT_DIR, "protected_region_analysis.csv"), index=False)

summary_exp = df_metrics.groupby("exp_name")[[
    "t_preprocess_s", "t_infer_s", "t_total_s", "peak_vram_gb", "mask_area_pct",
    "lpips_out_unified", "ssim_out_unified", "clip_cloth", "clip_gain", "dino_cloth", "dino_gain"
]].mean().round(4)
summary_exp.to_csv(os.path.join(OUT_DIR, "summary_by_config.csv"))

summary_grp = df_metrics[df_metrics["exp_name"].isin(["C0_baseline_s20_g45", "C1_refined_s20_g45", "C2_refined_s12_g25"])].groupby(["group", "exp_name"])[[
    "t_total_s", "mask_area_pct", "lpips_out_unified", "ssim_out_unified", "clip_gain", "dino_gain"
]].mean().round(4)
summary_grp.to_csv(os.path.join(OUT_DIR, "summary_by_group.csv"))

print("=== SUMMARY BY CONFIG ===")
print(summary_exp.to_string())
print("=== SUMMARY BY GROUP ===")
print(summary_grp.to_string())
print("=== PROTECTED REGION ===")
print(df_prot.to_string(index=False))

GRIDS_DIR = os.path.join(OUT_DIR, "grids")
os.makedirs(GRIDS_DIR, exist_ok=True)

def overlay_mask(person_img, mask_l, color_rgb):
    m = mask_l.convert("L").resize(person_img.size)
    mb = m.point(lambda v: 255 if v > 127 else 0)
    col = Image.new("RGB", person_img.size, color_rgb)
    return Image.composite(Image.blend(person_img, col, 0.52), person_img, mb)

pw, ph = 240, 320
for rec in MANIFEST["pairs"]:
    pid = rec["pair_id"]
    p_im = Image.open(os.path.join(TMP_ROOT, "C0_baseline_s20_g45", "person", f"{pid}.png")).convert("RGB")
    c_im = Image.open(os.path.join(TMP_ROOT, "C1_refined_s20_g45", "cloth", f"{pid}.png")).convert("RGB")
    m0 = Image.open(os.path.join(TMP_ROOT, "C0_baseline_s20_g45", "mask", f"{pid}.png"))
    m1 = Image.open(os.path.join(TMP_ROOT, "C1_refined_s20_g45", "mask", f"{pid}.png"))
    g0 = Image.open(os.path.join(TMP_ROOT, "C0_baseline_s20_g45", "gen", f"{pid}.png")).convert("RGB")
    g1 = Image.open(os.path.join(TMP_ROOT, "C1_refined_s20_g45", "gen", f"{pid}.png")).convert("RGB")
    g2 = Image.open(os.path.join(TMP_ROOT, "C2_refined_s12_g25", "gen", f"{pid}.png")).convert("RGB")

    panels = [
        ("Person", p_im),
        ("Cloth", c_im),
        ("Mask Base (C0)", overlay_mask(p_im, m0, (220, 38, 38))),
        ("Mask Refined (C1)", overlay_mask(p_im, m1, (37, 99, 235))),
        ("Out C0 Base", g0),
        ("Out C1 Refined", g1),
        ("Out C2 Fast12", g2),
    ]
    grid = Image.new("RGB", (pw * len(panels), ph + 24), (255, 255, 255))
    draw = ImageDraw.Draw(grid)
    for idx_p, (lbl, im) in enumerate(panels):
        draw.text((idx_p * pw + 8, 5), f"{pid} | {lbl}", fill=(20, 20, 20))
        grid.paste(im.resize((pw, ph), Image.LANCZOS), (idx_p * pw, 24))
    grid.save(os.path.join(GRIDS_DIR, f"{pid}_compare.jpg"), quality=88)

print("Da luu toan bo ket qua tai:", OUT_DIR)
